# 08 — Scan and compare the contract portfolio

In 07 we selected two PDFs. Here we discover **every top-level PDF in `contracts/`**, show why each can or cannot be calculated, and compare compatible supported plans.

This is an **unapproved demonstration using synthetic usage**, not a current-offer recommendation. The pricing adapter still supports only Reliant Power Savings 2,000 kWh 12/24 EFLs. Other PDFs remain visible for future adapters/review. No API key or live API calls are needed. Run sections in order using your `.venv` kernel.

## 1. Imports and project paths
This notebook is independent of previous kernel sessions. It reads the synthetic usage file saved by 04.

In [1]:
from pathlib import Path
from typing import TypedDict
from collections import Counter
from datetime import datetime, timezone
from html import escape
import json
from IPython.display import HTML, display
from langgraph.graph import StateGraph, START, END
from electricity_optimizer.portfolio import scan_portfolio, compare_portfolio
from electricity_optimizer.usage import load_usage_csv

ROOT = Path.cwd()
if not (ROOT / "contracts").is_dir():
    raise FileNotFoundError("Open this notebook from the project root containing contracts/.")
CONTRACTS = ROOT / "contracts"
EXTRACTIONS = ROOT / "output/extraction"
USAGE_PATH = ROOT / "output/lesson04_synthetic/SYNTHETIC_usage_2025.csv"
if not USAGE_PATH.exists():
    raise FileNotFoundError("Run Notebook 04 to save its synthetic usage CSV first.")
usage = load_usage_csv(USAGE_PATH)

def table(rows, columns):
    header = "".join(f"<th>{escape(c)}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(r.get(c, '')))}</td>" for c in columns) + "</tr>" for r in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

print("PDF folder:", CONTRACTS)
print("Synthetic annual kWh:", sum(m.kwh for m in usage.months))

PDF folder: c:\Users\Nyalo\VSCode_Projects\Electricity_Agreement_Optimizer\contracts
Synthetic annual kWh: 11970


## 2. Build the portfolio workflow

`scan → compare`: the first node reads PDFs and audits matching saved extractions; the second groups supported prices by territory, currency and document date. This graph uses Python functions, with no LLM calls.

Bad PDFs and malformed saved records are reported individually. Duplicate PDF bytes count once. A group needs at least two distinct plan names to rank.

In [2]:
class PortfolioState(TypedDict, total=False):
    scan: dict
    groups: list

def scan_node(state):
    return {"scan": scan_portfolio(CONTRACTS, EXTRACTIONS)}

def compare_node(state):
    return {"groups": compare_portfolio(state["scan"], usage)}

builder = StateGraph(PortfolioState)
builder.add_node("scan", scan_node)
builder.add_node("compare", compare_node)
builder.add_edge(START, "scan")
builder.add_edge("scan", "compare")
builder.add_edge("compare", END)
workflow = builder.compile()
result = workflow.invoke({})
print("PDFs inventoried:", len(result["scan"]["inventory"]))

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
PDFs inventoried: 7


## 3. Read the inventory

- **demo_ready**: the narrow adapter found its required source prices; not approved.
- **unsupported_or_incomplete**: no supported complete pricing representation. This does **not** mean there are no prices anywhere in the PDF.
- **needs_text_review / unreadable**: inspect the PDF or improve extraction first.
- **duplicate**: another filename contains the same bytes.

Extraction status is separate: missing or stale saved AI output does not prevent the independent source-price adapter from running.

In [3]:
inventory = result["scan"]["inventory"]
table(inventory, ["file", "status", "extraction_status", "reason"])
print(dict(Counter(row["status"] for row in inventory)))
if not inventory:
    print("No PDFs found. Add PDFs to contracts/ and rerun Section 2.")

file,status,extraction_status,reason
2106-00134_Terms_and_Conditions_of_Supply_-_1_June_2021.pdf,unsupported_or_incomplete,no_matching_saved_record,2106-00134_Terms_and_Conditions_of_Supply_-_1_June_2021.pdf: name missing or conflicting; inspect the source. Only the Power Savings 12/24 layout is supported; this does not prove the PDF lacks prices.
EA_MKTTC_DL_0224_FINAL_Digital.pdf,unsupported_or_incomplete,no_matching_saved_record,EA_MKTTC_DL_0224_FINAL_Digital.pdf: name missing or conflicting; inspect the source. Only the Power Savings 12/24 layout is supported; this does not prove the PDF lacks prices.
GetContractVersionPDF_143273.pdf,unsupported_or_incomplete,no_matching_saved_record,GetContractVersionPDF_143273.pdf: name missing or conflicting; inspect the source. Only the Power Savings 12/24 layout is supported; this does not prove the PDF lacks prices.
R1F00160000021A.pdf,demo_ready,saved_matching_unapproved,Labeled PDF prices parsed independently; not human-approved.
R1F00166003766B.pdf,demo_ready,saved_matching_unapproved,Labeled PDF prices parsed independently; not human-approved.
R1F00169972621A.pdf,unsupported_or_incomplete,saved_matching_unapproved,R1F00169972621A.pdf: name missing or conflicting; inspect the source. Only the Power Savings 12/24 layout is supported; this does not prove the PDF lacks prices.
TOSA.pdf,unsupported_or_incomplete,no_matching_saved_record,TOSA.pdf: name missing or conflicting; inspect the source. Only the Power Savings 12/24 layout is supported; this does not prove the PDF lacks prices.


{'unsupported_or_incomplete': 5, 'demo_ready': 2}


## 4. Inspect extraction issues and source prices
Existing model quotes are checked against the current PDF. Errors remain unresolved even when independent price parsing succeeds. Expand the printed evidence by choosing a filename below.

In [4]:
for row in inventory:
    if row["issues"]:
        print("\n", row["file"])
        table(row["issues"], ["severity", "field", "message"])
for error in result["scan"]["saved_record_errors"]:
    print("Saved-record read error:", error["file"], error["error"])

plans = result["scan"]["plans"]
SELECTED_FILE = plans[0].source_file if plans else None  # Change to another supported PDF filename.
selected = next((p for p in plans if p.source_file == SELECTED_FILE), None)
if selected:
    print("Source evidence:", selected.source_file)
    table([{"field": field, **e.model_dump()} for field, e in selected.evidence.items()],
          ["field", "value", "page_number", "quote"])
else:
    print("No supported plan selected.")


 R1F00160000021A.pdf


severity,field,message
warning,customer_type,Not stated: obtain supporting documents before relying on this term.
error,credits_and_minimum_usage,Quote not found on page 1.
warning,renewal_terms,Not stated: obtain supporting documents before relying on this term.



 R1F00166003766B.pdf


severity,field,message
warning,customer_type,Not stated: obtain supporting documents before relying on this term.
error,credits_and_minimum_usage,Quote not found on page 1.
warning,renewal_terms,Not stated: obtain supporting documents before relying on this term.



 R1F00169972621A.pdf


severity,field,message
warning,customer_type,Not stated: obtain supporting documents before relying on this term.
warning,recurring_charges,Not stated: obtain supporting documents before relying on this term.
warning,credits_and_minimum_usage,Not stated: obtain supporting documents before relying on this term.
warning,renewal_terms,Not stated: obtain supporting documents before relying on this term.


Source evidence: R1F00160000021A.pdf


field,value,page_number,quote
name,"Reliant Power Savings 2,000 kWh 24 plan",1,"Reliant Power Savings 2,000 kWh 24 plan"
market,AEP Texas Central,1,AEP Texas Central service area
document_date,09/01/2026,1,Date: 09/01/2026
energy_usd_per_kwh,0.154124,1,Energy Charge: 15.4124¢ per kWh
base_usd_per_month,0.00,1,Base Charge: $0.00 per billing cycle
delivery_usd_per_month,3.24,1,AEP Texas Central Delivery Charges: $3.24 per billing cycle and 5.7554¢ per kWh
delivery_usd_per_kwh,0.057554,1,AEP Texas Central Delivery Charges: $3.24 per billing cycle and 5.7554¢ per kWh
credit_usd,150.00,1,"A Usage Credit of $150.00 will be included for each billing cycle when your usage on this plan is above or equal to 2,000 kWh"
credit_min_kwh,2000,1,"A Usage Credit of $150.00 will be included for each billing cycle when your usage on this plan is above or equal to 2,000 kWh"
contract_months,24,1,Contract Term 24 months


## 5. Compare within each compatible group

Costs use the same twelve synthetic monthly usage values for every plan. Rates are held at the PDF snapshot, with inclusive monthly usage credits and component rounding to cents (half up). USD is the adapter's US-plan assumption. The comparison covers 12 months even for a 24-month commitment.

Taxes, deposits, early termination/switching charges, future tariff changes and special-territory adjustments are excluded. Standard AEP Texas Central delivery totals are used without separately estimating ITR adjustments. The McAllen/Mission former-Oncor exception is outside this demonstration. Matching group keys alone do not establish customer eligibility.

In [5]:
for group in result["groups"]:
    print("\n", group["market"], group["currency"], group["document_date"], "—", group["status"])
    if not group["results"]:
        print(group["reason"])
        continue
    table([{"plan": r["plan"]["name"], "12-month USD": r["annual_usd"],
            "contract months": r["plan"]["contract_months"],
            "termination USD (excluded)": r["plan"]["termination_usd"]}
           for r in group["results"]],
          ["plan", "12-month USD", "contract months", "termination USD (excluded)"])
if not result["groups"]:
    print("No supported groups yet; inspect the inventory reasons.")


 AEP Texas Central USD 09/01/2026 — demonstration_only


plan,12-month USD,contract months,termination USD (excluded)
"Reliant Power Savings 2,000 kWh 24 plan",2572.69,24,295
"Reliant Power Savings 2,000 kWh 12 plan",2720.59,12,150


## 6. Save the portfolio report
This saves a new lesson report with all inventory rows, source evidence, unresolved issues, assumptions and calculated bills. It does not modify your PDFs, extraction records or Notebook 06 review JSON.

In [6]:
report = {
    "status": "source_backed_demonstration_not_approved",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "synthetic_usage": True,
    "assumptions": ["12 months; snapshot rates held constant; USD assumed for supported US EFLs",
                    "Standard AEP Texas Central territory; excludes former-Oncor exception",
                    "Component rounding to cents, half up; inclusive monthly credit threshold",
                    "Excludes taxes, deposits, switching/termination costs and future tariff changes",
                    "Uses listed delivery totals without independently estimating ITR adjustments",
                    "AI evidence errors remain unresolved; no human approval is implied"],
    "usage": usage.model_dump(mode="json"),
    "inventory": inventory,
    "saved_record_errors": result["scan"]["saved_record_errors"],
    "supported_plans": [p.model_dump(mode="json") for p in plans],
    "groups": result["groups"],
}
destination = ROOT / "output/lesson08/portfolio_comparison.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved:", destination)

Saved: c:\Users\Nyalo\VSCode_Projects\Electricity_Agreement_Optimizer\output\lesson08\portfolio_comparison.json


## 7. What to try next

Add another PDF and rerun Sections 2–6. It should appear in the inventory even if it cannot be ranked. Check its exclusion reason before choosing a next pricing adapter.

For your existing seven PDFs, expect the two Power Savings plans to form one demonstration comparison group. The other documents remain listed for pricing support/review. A future lesson can add adapters for more pricing structures and a recommendation explanation grounded in the saved results.